# Assignment 2 - Recommender Systems (Simple Version)
**Anime domain (MyAnimeList)**


---
- **User Tower**: user_id embedding → 64-dim vector
- **Item Tower**: anime_id embedding + genre + numerical features → concat → Linear → 64-dim vector
- **Loss**: BPR (Bayesian Personalized Ranking)
- **Eval**: Recall@K

---
## 0. Install & Kaggle Setup

In [1]:
!pip install kaggle -q

Fill in your Kaggle credentials and run — no file upload needed.

In [2]:
import os, json

KAGGLE_USERNAME = 'your_username'    # Insert your Kaggle username
KAGGLE_KEY      = 'your_kaggle_key'  # Insert your Kaggle API key

os.makedirs('/root/.config/kaggle', exist_ok=True)
with open('/root/.config/kaggle/kaggle.json', 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured.')

Kaggle credentials configured.


In [3]:
!kaggle datasets download -d CooperUnion/anime-recommendations-database
!unzip -q anime-recommendations-database.zip
print('Dataset ready.')

Dataset URL: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database
License(s): CC0-1.0
100% 25.0M/25.0M [00:00<00:00, 68.1MB/s]

Dataset ready.


---
## 1. Imports & Hyperparameters

In [4]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder

SEED   = 42
DIM    = 64
BATCH  = 2048
EPOCHS = 10
LR     = 1e-2

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


---
## 2. Load Data

In [5]:
anime_df   = pd.read_csv('anime.csv')
ratings_df = pd.read_csv('rating.csv')

print(f'Anime: {len(anime_df):,}  |  Ratings: {len(ratings_df):,}')
anime_df.head(3)

Anime: 12,294  |  Ratings: 7,813,737


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262


---
## 3. Data Preprocessing

In [6]:
# Clean anime metadata
anime_df = anime_df.dropna(subset=['genre', 'type', 'rating']).reset_index(drop=True)

# --- Categorical: type (TV/Movie/OVA...) and primary genre ---
anime_df['primary_genre'] = anime_df['genre'].str.split(',').str[0].str.strip()
type_enc  = LabelEncoder().fit(anime_df['type'])
genre_enc = LabelEncoder().fit(anime_df['primary_genre'])
anime_df['type_idx']  = type_enc.transform(anime_df['type'])
anime_df['genre_idx'] = genre_enc.transform(anime_df['primary_genre'])

# --- Numerical: episodes, community rating, members ---
anime_df['episodes'] = pd.to_numeric(anime_df['episodes'], errors='coerce')
anime_df['episodes'] = anime_df['episodes'].fillna(anime_df['episodes'].median())
anime_df['episodes_log'] = np.log1p(anime_df['episodes'])
anime_df['rating_norm']  = anime_df['rating'] / 10.0
anime_df['members_log']  = np.log1p(anime_df['members'])

for c in ['episodes_log', 'rating_norm', 'members_log']:
    mn, mx = anime_df[c].min(), anime_df[c].max()
    anime_df[c] = (anime_df[c] - mn) / (mx - mn + 1e-9)

# --- Ratings: keep explicit only (drop -1), keep users with 20+ ratings ---
ratings_df = ratings_df[ratings_df['rating'] != -1].copy()
valid_ids  = set(anime_df['anime_id'].values)
ratings_df = ratings_df[ratings_df['anime_id'].isin(valid_ids)].reset_index(drop=True)

counts       = ratings_df.groupby('user_id').size()
active_users = counts[counts >= 20].index
ratings_df   = ratings_df[ratings_df['user_id'].isin(active_users)].reset_index(drop=True)

# Map users and anime to contiguous integer indices
users_all = sorted(ratings_df['user_id'].unique())
anime_all = sorted(ratings_df['anime_id'].unique())
u2i = {u: i for i, u in enumerate(users_all)}
a2i = {a: i for i, a in enumerate(anime_all)}
i2a = {i: a for a, i in a2i.items()}

N_USERS = len(users_all)
N_ANIME = len(anime_all)

ratings_df['uid'] = ratings_df['user_id'].map(u2i)
ratings_df['aid'] = ratings_df['anime_id'].map(a2i)

print(f'Users: {N_USERS:,}  |  Anime: {N_ANIME:,}  |  Interactions: {len(ratings_df):,}')

Users: 47,153  |  Anime: 9,885  |  Interactions: 6,164,892


---
## 4. Build Item Feature Tensors

In [7]:
anime_df = anime_df.set_index('anime_id')

item_type  = torch.zeros(N_ANIME, dtype=torch.long)
item_genre = torch.zeros(N_ANIME, dtype=torch.long)
item_num   = torch.zeros(N_ANIME, 3)   # episodes, rating, members

for aid, idx in a2i.items():
    if aid not in anime_df.index:
        continue
    row = anime_df.loc[aid]
    item_type[idx]  = int(row['type_idx'])
    item_genre[idx] = int(row['genre_idx'])
    item_num[idx]   = torch.tensor([row['episodes_log'], row['rating_norm'], row['members_log']])

item_type  = item_type.to(device)
item_genre = item_genre.to(device)
item_num   = item_num.to(device)

N_TYPES  = int(item_type.max().item()) + 1
N_GENRES = int(item_genre.max().item()) + 1
print(f'Types: {N_TYPES}  |  Genres: {N_GENRES}')

Types: 6  |  Genres: 40


---
## 5. Train / Test Split

In [8]:
perm     = np.random.permutation(len(ratings_df))
split    = int(len(ratings_df) * 0.8)
train_df = ratings_df.iloc[perm[:split]].reset_index(drop=True)
test_df  = ratings_df.iloc[perm[split:]].reset_index(drop=True)

# Per-user positive sets for negative sampling
user_pos = {}
for row in train_df.itertuples():
    user_pos.setdefault(row.uid, set()).add(row.aid)

test_users = list(test_df['uid'].values)
test_anime = list(test_df['aid'].values)

print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

Train: 4,931,913  |  Test: 1,232,979


---
## 6. Dataset with Negative Sampling

In [9]:
class BPRDataset(Dataset):
    def __init__(self, df):
        self.pairs = list(zip(df['uid'].values, df['aid'].values))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        u, pos = self.pairs[i]
        neg = random.randint(0, N_ANIME - 1)
        while neg in user_pos.get(u, set()):
            neg = random.randint(0, N_ANIME - 1)
        return u, pos, neg


train_loader = DataLoader(BPRDataset(train_df), batch_size=BATCH, shuffle=True, drop_last=True)

---
## 7. The Two-Tower Model

```
User Tower  : user_id → Embedding → 64-dim vector

Item Tower  : anime_id  → Embedding  ─┐
              type      → Embedding  ─┤→ concat → Linear → 64-dim vector
              genre     → Embedding  ─┤
              3 numbers → raw values ─┘

Score = dot(user_vec, item_vec)
```

In [10]:
class TwoTower(nn.Module):
    def __init__(self, n_users, n_anime, n_types, n_genres, dim):
        super().__init__()
        # User tower
        self.user_emb  = nn.Embedding(n_users, dim)

        # Item tower — one embedding per feature type
        self.anime_emb = nn.Embedding(n_anime,  dim)
        self.type_emb  = nn.Embedding(n_types,  dim // 4)
        self.genre_emb = nn.Embedding(n_genres, dim // 4)

        # Project concat(anime_emb, type_emb, genre_emb, 3 numerics) → dim
        in_dim = dim + dim // 4 + dim // 4 + 3
        self.item_proj = nn.Linear(in_dim, dim)

        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.anime_emb.weight)
        nn.init.xavier_uniform_(self.item_proj.weight)

    def item_vector(self, aid):
        x = torch.cat([
            self.anime_emb(aid),
            self.type_emb(item_type[aid]),
            self.genre_emb(item_genre[aid]),
            item_num[aid],
        ], dim=-1)
        return self.item_proj(x)

    def forward(self, uid, pos, neg):
        u = self.user_emb(uid)
        p = self.item_vector(pos)
        n = self.item_vector(neg)
        return u, p, n


model = TwoTower(N_USERS, N_ANIME, N_TYPES, N_GENRES, DIM).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=LR)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

Parameters: 3,657,568


---
## 8. Evaluation

In [11]:
@torch.no_grad()
def evaluate(ks=(1, 5, 10, 50, 100)):
    model.eval()

    all_ids  = torch.arange(N_ANIME, device=device)
    all_vecs = model.item_vector(all_ids)    # pre-compute all item vectors

    hits  = {k: 0 for k in ks}
    total = len(test_users)

    t_users = torch.tensor(test_users, device=device)
    t_anime = torch.tensor(test_anime, device=device)

    for start in range(0, total, 4096):
        end    = min(start + 4096, total)
        u_emb  = model.user_emb(t_users[start:end])
        scores = u_emb @ all_vecs.T
        true_a = t_anime[start:end]

        for k in ks:
            topk = scores.topk(k, dim=1).indices
            hits[k] += (topk == true_a.unsqueeze(1)).any(1).sum().item()

    return {k: hits[k] / total for k in ks}

---
## 9. Training Loop

In [12]:
from tqdm.auto import tqdm

header = f"{'Epoch':>6}  {'Loss':>8}  {'Recall@1':>9}  {'Recall@5':>9}  {'Recall@10':>10}  {'Recall@50':>10}  {'Recall@100':>11}"
print(header)
print('-' * len(header))

for ep in range(1, EPOCHS + 1):
    model.train()
    total_loss, steps = 0.0, 0

    loop = tqdm(train_loader, desc=f'Epoch {ep}/{EPOCHS}', leave=False)
    for uid, pos, neg in loop:
        uid = uid.to(device); pos = pos.to(device); neg = neg.to(device)

        u, p, n   = model(uid, pos, neg)
        pos_score = (u * p).sum(1)
        neg_score = (u * n).sum(1)
        loss      = -F.logsigmoid(pos_score - neg_score).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()

        total_loss += loss.item()
        steps += 1
        loop.set_postfix(loss=f'{total_loss/steps:.4f}')

    m = evaluate()
    print(f"{'Epoch '+str(ep):>6}  {total_loss/steps:>8.4f}  "
          f"{m[1]:>9.2%}  {m[5]:>9.2%}  {m[10]:>10.2%}  {m[50]:>10.2%}  {m[100]:>11.2%}")

m = evaluate()
print(f"""
Final Results
-------------
Recall@1   :  {m[1]:.2%}
Recall@5   :  {m[5]:.2%}
Recall@10  :  {m[10]:.2%}
Recall@50  :  {m[50]:.2%}
Recall@100 :  {m[100]:.2%}
""")

 Epoch      Loss   Recall@1   Recall@5   Recall@10   Recall@50   Recall@100
---------------------------------------------------------------------------


Epoch 1/10:   0%|          | 0/2408 [00:00<?, ?it/s]

Epoch 1    0.1721      0.46%      2.11%       3.88%      14.36%       23.62%


Epoch 2/10:   0%|          | 0/2408 [00:00<?, ?it/s]

Epoch 2    0.1332      0.47%      2.09%       3.87%      14.77%       24.56%


...

Epoch 8    0.1206      0.47%      2.14%       4.01%      15.52%       25.82%


Epoch 9/10:   0%|          | 0/2408 [00:00<?, ?it/s]

Epoch 9    0.1203      0.45%      2.07%       3.88%      15.07%       25.28%


Epoch 10/10:   0%|          | 0/2408 [00:00<?, ?it/s]

Epoch 10    0.1200      0.44%      2.06%       3.88%      14.97%       25.13%

Final Results
-------------
Recall@1   :  0.44%
Recall@5   :  2.06%
Recall@10  :  3.88%
Recall@50  :  14.97%
Recall@100 :  25.13%



---
## 10. Make Recommendations

In [13]:
id2name = anime_df['name'].to_dict()


@torch.no_grad()
def recommend(user_idx, k=10, exclude_seen=True):
    model.eval()

    uid_t    = torch.tensor([user_idx], device=device)
    u_vec    = model.user_emb(uid_t)                          # (1, DIM)
    all_ids  = torch.arange(N_ANIME, device=device)
    all_vecs = model.item_vector(all_ids)                     # (N_ANIME, DIM)
    scores   = (u_vec @ all_vecs.T).squeeze(0).cpu()

    if exclude_seen:
        for aid in user_pos.get(user_idx, []):
            scores[aid] = -1e9

    top_k = scores.argsort(descending=True)[:k].tolist()
    return [(id2name.get(i2a[aid], f'id:{i2a[aid]}'), scores[aid].item()) for aid in top_k]


for uid in range(5):
    print(f'\nTop 10 for user {uid}:')
    for i, (name, score) in enumerate(recommend(uid, k=10), 1):
        print(f'  {i:>2}. {score:+.3f}  {name}')


Top 10 for user 0:
   1. +29.573  Sword Art Online
   2. +29.360  Diabolik Lovers
   3. +28.380  Free!
   4. +28.167  Sword Art Online II
   5. +27.549  Owari no Seraph: Nagoya Kessen-hen
   6. +27.429  Death Note
   7. +27.405  Kuroshitsuji: Book of Circus
   8. +27.226  Prison School
   9. +27.084  Fairy Tail OVA
  10. +27.058  Mirai Nikki (TV)

Top 10 for user 1:
   1. +19.293  Dash! Kappei
   2. +18.580  Bakemonogatari
   3. +18.484  Recorder to Randoseru Re♪
   4. +18.445  Mirai Nikki (TV)
   5. +18.324  Baka to Test to Shoukanjuu Ni!
   6. +18.306  Amagami SS+ Plus
   7. +18.208  Kokoro Connect
   8. +18.028  Clannad
   9. +17.962  Yahari Ore no Seishun Love Comedy wa Machigatteiru.
  10. +17.936  Dragon Ball

Top 10 for user 2:
   1. +24.391  Tokyo Ghoul
   2. +24.347  Shingeki no Kyojin
   3. +24.127  Yahari Ore no Seishun Love Comedy wa Machigatteiru. Zoku
   4. +23.672  Amagi Brilliant Park
   5. +23.485  Corpse Party: Tortured Souls - Bougyakusareta Tamashii no Jukyou
   6.